In [ ]:
from pathlib import Path

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
)
from conversational_toolkit.embeddings.qwen_vl import Qwen3VLEmbeddings
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import EMBEDDING_MODEL
from loguru import logger
from dotenv import load_dotenv

In [2]:
load_dotenv("../../.env.local")

_DB_DIR = Path("../src/sme_kt_zh_collaboration_rag/db")

IMAGE_VS_PATH = _DB_DIR / "vs_image"
TEXT_VS_PATH = _DB_DIR / "vs_text"

logger.info("Initializing vector stores...")
all_chunks = load_chunks()

2026-03-18 13:44:35.026 | INFO     | __main__:<module>:8 - Initializing vector stores...
2026-03-18 13:44:35.027 | WARNING  | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:199 - Skipping unsupported file type '': .DS_Store
2026-03-18 13:44:35.028 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:206 - Chunking 34 files from /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/data
objc[62119]: Class AVFFrameReceiver is implemented in both /Users/pkoerner/Desktop/Kanton_Zurich/rag_venv/lib/python3.13/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x12d1943a8) and /Users/pkoerner/Desktop/Kanton_Zurich/rag_venv/lib/python3.13/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x14308c3a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[62119]: Class AVFAudioReceiver is implemented in both /Users/pkoerner/Desktop/Kanton_Zurich/rag_venv/lib/python3.13/site-

In [3]:
text_chunks = [c for c in all_chunks if c.mime_type.startswith("text/")]
# limit content size to 8k tokens for OpenAI embedding model
text_chunks = [c for c in text_chunks if len(c.content) < 8 * 1024]
await build_vector_store(
    text_chunks,
    OpenAIEmbeddings(model_name=EMBEDDING_MODEL),
    db_path=TEXT_VS_PATH,
    reset=True,
)

2026-03-18 13:51:33.527 | DEBUG    | conversational_toolkit.embeddings.openai:__init__:20 - OpenAI embeddings model loaded: text-embedding-3-small
2026-03-18 13:51:33.640 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:265 - Reset vector store collection at ../src/sme_kt_zh_collaboration_rag/db/vs_text
2026-03-18 13:51:33.641 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:273 - Embedding 1635 chunks with <conversational_toolkit.embeddings.openai.OpenAIEmbeddings object at 0x1472e8ec0> ...
2026-03-18 13:51:36.241 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (10, 1024)
2026-03-18 13:51:36.268 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:282 - Processed batch 1/164
2026-03-18 13:51:36.501 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (10, 1024)
2026-03-18 13:51:36.517 | INFO     

In [5]:
image_chunks = [c for c in all_chunks if c.mime_type.startswith("image/")]
await build_vector_store(
    image_chunks,
    Qwen3VLEmbeddings(device="cpu"),
    db_path=IMAGE_VS_PATH,
    reset=True,
    batch_size=1,
)

2026-03-18 14:03:37.999 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:265 - Reset vector store collection at ../src/sme_kt_zh_collaboration_rag/db/vs_image
2026-03-18 14:03:38.011 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:273 - Embedding 934 chunks with <conversational_toolkit.embeddings.qwen_vl.Qwen3VLEmbeddings object at 0x358dc3890> ...
2026-03-18 14:04:28.724 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:282 - Processed batch 1/934
2026-03-18 14:04:53.639 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:282 - Processed batch 2/934
2026-03-18 14:05:12.289 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:282 - Processed batch 3/934
2026-03-18 14:05:33.846 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:282 - Processed batch 4/934
2026-03-18 14:06:04.537 | INFO     | sme_kt_zh